# Results KPI Monitor

Purpose: fast daily health check (PASS/WARN) from latest scorecard artifacts.

This is the morning checkpoint, not the root-cause notebook.

Fast daily scan for model health, WARN state, and operator action.

In [1]:
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

ODDS_DIR = ROOT / "artifacts" / "odds_log"
SCORECARD_PATH = ODDS_DIR / "model_health_scorecard_daily.parquet"
GATE_PATH = ODDS_DIR / "gate_next_n_comparison.parquet"


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas().round(3)
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


print("repo:", ROOT)
print("scorecard:", SCORECARD_PATH)
print("gate:", GATE_PATH)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
scorecard: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\model_health_scorecard_daily.parquet
gate: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\gate_next_n_comparison.parquet


In [2]:
if not SCORECARD_PATH.exists():
    print("Missing scorecard artifact. Run results dashboard scorecard section first.")
else:
    score = pl.read_parquet(SCORECARD_PATH).sort("snapshot_utc")
    latest = score.tail(1)
    cols = [
        c
        for c in [
            "snapshot_utc",
            "n_joined",
            "n_warn",
            "mae_err_k_rate",
            "under_bias_tbf",
            "worst_matchup_tier_mae_err_k_rate",
            "long_rest_bias_tbf",
        ]
        if c in latest.columns
    ]
    print("latest scorecard row")
    show_table(latest.select(cols))
    print("\nrecent trend (last 10)")
    trend_cols = [c for c in ["snapshot_utc", "n_warn", "mae_err_k_rate", "under_bias_tbf"] if c in score.columns]
    show_table(score.select(trend_cols).tail(10))

latest scorecard row


,snapshot_utc,n_joined,n_warn,mae_err_k_rate,under_bias_tbf,long_rest_bias_tbf
0,2026-08-18T04:43:29.550943+00:00,743,3,0.078,-1.081,-8.582



recent trend (last 10)


,snapshot_utc,n_warn,mae_err_k_rate,under_bias_tbf
0,2026-08-07T17:54:23.636635+00:00,3,0.073,-2.516
1,2026-08-10T16:12:47.313541+00:00,4,0.081,-1.532
2,2026-08-11T14:58:57.251535+00:00,3,0.079,-1.433
3,2026-08-14T20:19:17.334272+00:00,3,0.078,-1.047
4,2026-08-14T22:23:11.589546+00:00,3,0.078,-1.047
5,2026-08-14T23:29:20.805747+00:00,3,0.078,-1.047
6,2026-08-15T00:05:56.272442+00:00,3,0.078,-1.047
7,2026-08-15T05:58:50.158753+00:00,3,0.078,-1.047
8,2026-08-15T06:07:06.696469+00:00,3,0.078,-1.047
9,2026-08-18T04:43:29.550943+00:00,3,0.078,-1.081


In [3]:
if not GATE_PATH.exists():
    print("No gate comparison artifact yet.")
else:
    gate = pl.read_parquet(GATE_PATH).sort("snapshot_utc")
    cols = [c for c in ["snapshot_utc", "next_n", "gate_pnl_delta", "gate_clv_delta_pp", "gate_bet_count_delta"] if c in gate.columns]
    show_table(gate.select(cols).tail(10))

,snapshot_utc,gate_pnl_delta
0,2026-08-17T15:28:40.601573+00:00,0.0
1,2026-08-17T21:39:53.057495+00:00,0.0
2,2026-08-18T07:00:08.367959+00:00,0.0
3,2026-08-18T14:34:56.244327+00:00,0.0
4,2026-08-18T14:36:08.368690+00:00,0.0
5,2026-08-19T17:18:20.720457+00:00,0.0
6,2026-08-19T17:22:56.743614+00:00,0.0
7,2026-08-19T17:30:24.545663+00:00,0.0
8,2026-08-20T07:00:28.675327+00:00,0.0
9,2026-08-20T12:31:21.310727+00:00,0.0


In [ ]:
# Compact governance + risk snapshot (daily quick scan)
import json

SUMMARY_PATH = ODDS_DIR / "daily_operator_summary.json"
REPLAY_PATH = ODDS_DIR / "policy_replay_daily.json"
RANKED_GOV_PATH = ODDS_DIR / "feature_set_governance_ranked.csv"

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}
replay = json.loads(REPLAY_PATH.read_text(encoding="utf-8")) if REPLAY_PATH.exists() else {}
ranked_gov = pl.read_csv(RANKED_GOV_PATH) if RANKED_GOV_PATH.exists() else pl.DataFrame()

current_policy = None
for s in replay.get("scenarios", []):
    if isinstance(s, dict) and s.get("scenario") == "current_policy_flat_1u":
        current_policy = s
        break

top_candidate = ranked_gov.head(1).to_dicts()[0] if not ranked_gov.is_empty() else {}

rows = [
    {
        "panel": "current_policy_replay",
        "n": current_policy.get("n") if current_policy else None,
        "roi": current_policy.get("roi") if current_policy else None,
        "clv_mean_pp": current_policy.get("clv_mean_pp") if current_policy else None,
        "geo_growth_log_mean": current_policy.get("geo_growth_log_mean") if current_policy else None,
        "mc_prob_drawdown_breach": current_policy.get("mc_prob_drawdown_breach") if current_policy else None,
    },
    {
        "panel": "top_feature_governance_candidate",
        "n": top_candidate.get("n_policy_bets"),
        "roi": top_candidate.get("roi"),
        "clv_mean_pp": top_candidate.get("clv_mean_pp"),
        "geo_growth_log_mean": None,
        "mc_prob_drawdown_breach": None,
    },
]

print("Quick context (policy now vs top candidate):")
show_table(pl.DataFrame(rows), max_rows=10)

In [ ]:
# Regime-mix warning helper: compare recent settled windows
LEDGER_PATH = ODDS_DIR / "ledger.parquet"
if not LEDGER_PATH.exists():
    print(f"Missing {LEDGER_PATH}")
else:
    led = pl.read_parquet(LEDGER_PATH)
    settled = led.filter(
        (pl.col("status") == "settled")
        & (pl.col("stake").cast(pl.Float64).fill_null(0) > 0)
    ).sort("game_date")
    if settled.is_empty():
        print("No settled rows.")
    else:
        cuts = [("full", None), ("last_60", 60), ("last_30", 30)]
        out = []
        for label, n in cuts:
            s = settled if n is None else settled.tail(n)
            stake = float(s["stake"].cast(pl.Float64).sum())
            pnl = float(s["pnl"].cast(pl.Float64).sum())
            out.append(
                {
                    "window": label,
                    "n": int(s.height),
                    "roi": (pnl / stake) if stake > 0 else None,
                    "mean_clv_pp": float(s["clv_pp"].cast(pl.Float64).mean()) if "clv_pp" in s.columns and s.height else None,
                }
            )
        print("Regime-mix sanity check (prefer recent windows for current decisions):")
        show_table(pl.DataFrame(out), max_rows=10)